In [1]:
import pyspark
from pyspark.sql import SparkSession
pyspark.__version__

'3.5.1'

In [2]:
spark = SparkSession.builder.appName('sql').getOrCreate()

In [4]:
fifa = spark.read.csv('fifa19.csv',inferSchema=True,header=True)\
.select('Name','Age','Nationality','Club','Overall')

In [5]:
fifa.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Club: string (nullable = true)
 |-- Overall: integer (nullable = true)



In [11]:
print(type(fifa.limit(10).toPandas()))
# DataFrame.toPandas(): pandas의 DataFrame으로 변환
print(type(fifa))

<class 'pandas.core.frame.DataFrame'>
<class 'pyspark.sql.dataframe.DataFrame'>


In [13]:
df = fifa.withColumnRenamed('Name','Full_Name').withColumnRenamed('Club','Club_Name')
df.printSchema()

root
 |-- Full_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Club_Name: string (nullable = true)
 |-- Overall: integer (nullable = true)



In [14]:
df.createOrReplaceTempView('tmpview')

In [17]:
# AGE가 35보다 큰 선수 행의 전체열 추출
spark.sql("SELECT * FROM tmpview WHERE Age>35").show(5)

+--------------+---+-----------+--------------------+-------+
|     Full_Name|Age|Nationality|           Club_Name|Overall|
+--------------+---+-----------+--------------------+-------+
|     G. Buffon| 40|      Italy| Paris Saint-Germain|     88|
|Z. Ibrahimović| 36|     Sweden|           LA Galaxy|     85|
|   A. Barzagli| 37|      Italy|            Juventus|     84|
|   David Villa| 36|      Spain|    New York City FC|     82|
|        Aduriz| 37|      Spain|Athletic Club de ...|     82|
+--------------+---+-----------+--------------------+-------+
only showing top 5 rows



In [18]:
# AGE가 35보다 큰 Spain 선수 행의 전체열 추출
spark.sql("SELECT * FROM tmpview WHERE Age>35 AND Nationality=='Spain'").show(5)

+-----------+---+-----------+--------------------+-------+
|  Full_Name|Age|Nationality|           Club_Name|Overall|
+-----------+---+-----------+--------------------+-------+
|David Villa| 36|      Spain|    New York City FC|     82|
|     Aduriz| 37|      Spain|Athletic Club de ...|     82|
|   Casillas| 37|      Spain|            FC Porto|     82|
|    Joaquín| 36|      Spain|          Real Betis|     81|
|Diego López| 36|      Spain|        RCD Espanyol|     80|
+-----------+---+-----------+--------------------+-------+
only showing top 5 rows



In [29]:
# 각 Club_Name에 대해 선수들의 Overall 평균
# -> Club_Name과 Overall 평균만 출력
# 1. 각 Club_Name으로 grouping (GROUP BY)
club_overall = spark.sql("SELECT Club_Name, MEAN(Overall) AS Club_Overall \
FROM tmpview GROUP BY Club_Name")
club_overall.show(5)

+--------------------+------------------+
|           Club_Name|      Club_Overall|
+--------------------+------------------+
|             Palermo| 67.96666666666667|
|          Göztepe SK| 68.42857142857143|
|CD Everton de Viñ...|63.666666666666664|
|     Shonan Bellmare|              60.6|
|          Sagan Tosu| 64.23333333333333|
+--------------------+------------------+
only showing top 5 rows



In [24]:
# Club_Overall이 가장 큰 5개의 팀에 대한 Club_Name과 Club_Overall 출력
# orderBy의 내림차순 사용
club_overall.orderBy(club_overall['Club_Overall'].desc()).show(5)

+-----------+-----------------+
|  Club_Name|     Club_Overall|
+-----------+-----------------+
|   Juventus|            82.28|
|     Napoli|             80.0|
|      Inter|            79.75|
|Real Madrid|78.24242424242425|
|      Milan|78.07407407407408|
+-----------+-----------------+
only showing top 5 rows



In [25]:
from pyspark.ml.feature import SQLTransformer

In [26]:
sql_tf = SQLTransformer(statement="SELECT Club_Name FROM __THIS__")
type(sql_tf)

pyspark.ml.feature.SQLTransformer

In [28]:
sql_tf.transform(df).show(5)
sql_tf.transform(club_overall).show(5)

+-------------------+
|          Club_Name|
+-------------------+
|       FC Barcelona|
|           Juventus|
|Paris Saint-Germain|
|  Manchester United|
|    Manchester City|
+-------------------+
only showing top 5 rows

+--------------------+
|           Club_Name|
+--------------------+
|             Palermo|
|          Göztepe SK|
|CD Everton de Viñ...|
|     Shonan Bellmare|
|          Sagan Tosu|
+--------------------+
only showing top 5 rows



In [34]:
# SQLTransformer를 이용하여
# 각 Nationality에 대해 선수의 수와 Overall 평균을 보이시오.
# COUNT, MEAN
sql_tf2 = SQLTransformer(statement =\
                         "SELECT Nationality, COUNT(Nationality) AS Players, \
                         MEAN(Overall) AS Overall FROM __THIS__ GROUP BY Nationality")
national_overall = sql_tf2.transform(df)
national_overall.orderBy(national_overall.Players.desc()).show(5)

+-----------+-------+-----------------+
|Nationality|Players|          Overall|
+-----------+-------+-----------------+
|    England|   1662|63.42960288808664|
|    Germany|   1198|66.08681135225376|
|      Spain|   1072|69.69869402985074|
|  Argentina|    937|68.57203842049093|
|     France|    914|67.79868708971554|
+-----------+-------+-----------------+
only showing top 5 rows



In [37]:
sql_tf3 = SQLTransformer(statement =\
                         "SELECT Full_Name, Age, Overall \
                         FROM __THIS__ WHERE Age>40")
sql_tf3.transform(df).show(10)

+-------------+---+-------+
|    Full_Name|Age|Overall|
+-------------+---+-------+
|    J. Villar| 41|     77|
|     B. Nivet| 41|     71|
|     O. Pérez| 45|     71|
|     C. Muñoz| 41|     68|
|  S. Narazaki| 42|     65|
| H. Sulaimani| 41|     63|
|     M. Tyler| 41|     59|
|    T. Warner| 44|     53|
|K. Pilkington| 44|     48|
+-------------+---+-------+



In [38]:
from pyspark.sql.functions import expr

In [47]:
# withColumn + expr: 변환으로 생성된 열 추가
df.withColumn("Value",expr("Overall-Age")).show(10)
df.withColumn("Name_Overall",expr("CONCAT(Full_Name,'_',Overall)")).show(10)

+-----------------+---+-----------+-------------------+-------+-----+
|        Full_Name|Age|Nationality|          Club_Name|Overall|Value|
+-----------------+---+-----------+-------------------+-------+-----+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|   63|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|   61|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|   66|
|           De Gea| 27|      Spain|  Manchester United|     91|   64|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|   64|
|        E. Hazard| 27|    Belgium|            Chelsea|     91|   64|
|        L. Modrić| 32|    Croatia|        Real Madrid|     91|   59|
|        L. Suárez| 31|    Uruguay|       FC Barcelona|     91|   60|
|     Sergio Ramos| 32|      Spain|        Real Madrid|     91|   59|
|         J. Oblak| 25|   Slovenia|    Atlético Madrid|     90|   65|
+-----------------+---+-----------+-------------------+-------+-----+
only showing top 10 

In [51]:
# select + expr: where/filter와 유사한 기능 + 열 생성
df.select("*",expr('Overall>80 AND Age<30 AS High_Value')).show(10)

+-----------------+---+-----------+-------------------+-------+----------+
|        Full_Name|Age|Nationality|          Club_Name|Overall|High_Value|
+-----------------+---+-----------+-------------------+-------+----------+
|         L. Messi| 31|  Argentina|       FC Barcelona|     94|     false|
|Cristiano Ronaldo| 33|   Portugal|           Juventus|     94|     false|
|        Neymar Jr| 26|     Brazil|Paris Saint-Germain|     92|      true|
|           De Gea| 27|      Spain|  Manchester United|     91|      true|
|     K. De Bruyne| 27|    Belgium|    Manchester City|     91|      true|
|        E. Hazard| 27|    Belgium|            Chelsea|     91|      true|
|        L. Modrić| 32|    Croatia|        Real Madrid|     91|     false|
|        L. Suárez| 31|    Uruguay|       FC Barcelona|     91|     false|
|     Sergio Ramos| 32|      Spain|        Real Madrid|     91|     false|
|         J. Oblak| 25|   Slovenia|    Atlético Madrid|     90|      true|
+-----------------+---+--

In [54]:
# DataFrame.collect()
# cluster에 분산된 data를 master node로 모으는 작업을 수행
# 주의사항: master node의 memory 부족으로 오류가 발생할 수 있다.
#          주로 filter 등과 같이 data를 처리한 결과를 모아서 확인할 때 사용한다.
# collect는 Row 객체로 구성된 list를 반환
type(fifa.collect())
fifa.collect()

[Row(Name='L. Messi', Age=31, Nationality='Argentina', Club='FC Barcelona', Overall=94),
 Row(Name='Cristiano Ronaldo', Age=33, Nationality='Portugal', Club='Juventus', Overall=94),
 Row(Name='Neymar Jr', Age=26, Nationality='Brazil', Club='Paris Saint-Germain', Overall=92),
 Row(Name='De Gea', Age=27, Nationality='Spain', Club='Manchester United', Overall=91),
 Row(Name='K. De Bruyne', Age=27, Nationality='Belgium', Club='Manchester City', Overall=91),
 Row(Name='E. Hazard', Age=27, Nationality='Belgium', Club='Chelsea', Overall=91),
 Row(Name='L. Modrić', Age=32, Nationality='Croatia', Club='Real Madrid', Overall=91),
 Row(Name='L. Suárez', Age=31, Nationality='Uruguay', Club='FC Barcelona', Overall=91),
 Row(Name='Sergio Ramos', Age=32, Nationality='Spain', Club='Real Madrid', Overall=91),
 Row(Name='J. Oblak', Age=25, Nationality='Slovenia', Club='Atlético Madrid', Overall=90),
 Row(Name='R. Lewandowski', Age=29, Nationality='Poland', Club='FC Bayern München', Overall=90),
 Row(Nam

In [56]:
# list index를 통해 특정 data value를 추출할 때 사용할 수 있다.
print(fifa.collect()[10])
print(fifa.collect()[10]['Age'])

Row(Name='R. Lewandowski', Age=29, Nationality='Poland', Club='FC Bayern München', Overall=90)
29


In [64]:
# fifa DataFrame에서 Age가 가장 큰 선수의 Full_Name과 
# Age가 가장 작은 선수의 Full_Name의 value를 가져옴
youngest_name = fifa.orderBy('Age').collect()[0]['Name']
# youngest_name = fifa.orderBy('Age').collect()[0][0]
oldest_name = fifa.orderBy('Age').collect()[-1]['Name']
print(youngest_name)
print(oldest_name)

W. Geubbels
O. Pérez
